In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_silver = spark.table(f"{catalogo}.{esquema_source}.transacciones_clientes_silver")

In [0]:
df_silver_clean = df_silver.dropna(how="all")\
                           .filter((col("transaccion_id").isNotNull()) | (col("cliente_id").isNotNull()))

In [0]:
df_aggregated = df_silver_clean.groupBy("departamento", "banco_principal").agg(
    F.count("transaccion_id").alias("total_transacciones"),
    F.sum("monto_operacion").alias("monto_total_operado"),
    F.max("monto_operacion").alias("monto_maximo"),
    F.avg("score_riesgo").alias("promedio_score_riesgo")
)

In [0]:
df_with_diff = df_silver_clean.withColumn(
    "diferencia_monto", 
    F.abs(df_silver_clean["monto_operacion"] - df_silver_clean["monto_promedio_transaccion"]).cast(IntegerType())
)

In [0]:
df_updated = df_with_diff.select(
    "*",
    when(
        (col("alerta_sistema") == 1) | 
        (col("fraude_confirmado") == 1) | 
        (coalesce(col("score_riesgo"), lit(0)) > 0.6), 
        lit("Alerta Alta")
    ).otherwise(lit("Normal")).alias("prioridad_atencion"),
    
    when(
        (col("fraude_confirmado") == 1) | 
        (col("alerta_sistema") == 1) | 
        ((col("monto_operacion") > col("ingreso_mensual")) & (col("ubicacion_inusual") == 1)), 
        lit("Revisión Inmediata")
    ).otherwise(lit("Sin Riesgo")).alias("estado_fraude")
)

In [0]:
df_gold_final = df_updated.withColumn("ingestion_date_gold", current_timestamp())

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.golden_analisis_fraude")

print(" ¡Capa Golden unificada cargada exitosamente en Unity Catalog!")

In [0]:
# DBTITLE 1, Muestreo de la Tabla Final
df_gold_vista = df_gold_final.select(
    col("transaccion_id"),
    col("departamento"),
    col("banco_principal").alias("banco"),
    col("monto_operacion").alias("monto_soles"),
    col("score_riesgo"),
    when(
        (col("fraude_confirmado") == 1) | 
        (col("alerta_sistema") == 1) | 
        (coalesce(col("score_riesgo"), lit(0)) > 0.6), 
        lit("Alto")
    ).when(
        (col("ubicacion_inusual") == 1) | 
        (col("dispositivo_nuevo") == 1) |
        (coalesce(col("score_riesgo"), lit(0)) > 0.3), 
        lit("Medio")
    ).otherwise(lit("Bajo")).alias("nivel_riesgo"),
    col("prioridad_atencion"),
    col("estado_fraude"),
    col("ingestion_date_gold")
).orderBy(col("estado_fraude").desc(), col("prioridad_atencion").desc())

display(df_gold_vista)
print(f"Total de registros en la vista Gold: {df_gold_vista.count()}")